# Preprocessing Pipeline Walkthrough
Visualizing each step of our preprocessing pipeline.

In [ ]:
import os
import cv2
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path so we can import our modules
sys.path.append(os.path.abspath('..'))
from src.preprocessing.image_preprocessor import (
    load_image, skull_strip, resize_image, apply_clahe, denoise_image, normalize_image
)

## 1. Load a Sample Image

In [ ]:
data_dir = Path('../data/raw/Training')
sample_class = 'glioma'
sample_image_path = list((data_dir / sample_class).glob('*.jpg'))[0]

img_orig = load_image(sample_image_path, grayscale=True)

plt.imshow(img_orig, cmap='gray')
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. Step-by-Step Preprocessing

In [ ]:
img_skull_stripped = skull_strip(img_orig.copy())
img_resized = resize_image(img_skull_stripped, size=(224, 224))
img_clahe = apply_clahe(img_resized)
img_denoised = denoise_image(img_clahe)
img_normalized = normalize_image(img_denoised)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img_orig, cmap='gray')
axes[0, 0].set_title('1. Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_skull_stripped, cmap='gray')
axes[0, 1].set_title('2. Skull Stripped')
axes[0, 1].axis('off')

axes[0, 2].imshow(img_resized, cmap='gray')
axes[0, 2].set_title('3. Resized (224x224)')
axes[0, 2].axis('off')

axes[1, 0].imshow(img_clahe, cmap='gray')
axes[1, 0].set_title('4. CLAHE')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_denoised, cmap='gray')
axes[1, 1].set_title('5. Denoised (Gaussian)')
axes[1, 1].axis('off')

axes[1, 2].imshow(img_normalized, cmap='gray')
axes[1, 2].set_title('6. Normalized [0, 1]')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 3. Data Augmentation Demo

In [ ]:
from src.preprocessing.augmentation import get_training_augmentation

transform = get_training_augmentation()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img_orig, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 4):
    # Note: albumentations works with uint8 arrays typically, so apply before normalization ideally
    # Here we show it on the clahe/denoised version
    augmented = transform(image=img_denoised)
    # The transform returns a PyTorch tensor (C, H, W)
    aug_img = augmented['image'].numpy()[0] 
    axes[i].imshow(aug_img, cmap='gray')
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.show()